# Data modeling and query optimization

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-09-modeling-and-optimization.ipynb)

Two decisions shape almost every analytics query: how the data is modeled, and how the engine executes the query.
Good modeling makes the common query a simple join. Good optimization makes the join cheap. This notebook
covers both, using Spark's own plans rather than guesses.

Previous: [time-series analysis](sql-08-time-series.ipynb). Next: [interview drills](sql-10-interview-drills.ipynb).

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## Start with the grain

Before you model anything, write the grain as one sentence: "one row per order", "one row per customer per
day", "one row per segment change". Every table, join and aggregate must agree with that sentence. Most modeling
mistakes are a fact table whose grain is not written down.

A **star schema** keeps one **fact** table at a declared grain, surrounded by **dimension** tables that describe
its keys. Each dimension key must be unique, or every join to it fans out.

In [ ]:
create_view(
    "customers",
    "customer_id INT, name STRING, region STRING, signup_date DATE",
    [
        (1, "Acme Corp", "North", date(2022, 1, 5)),
        (2, "Beta LLC", "North", date(2022, 2, 10)),
        (3, "Cyan Inc", "South", date(2022, 3, 15)),
        (4, "Delta Co", "South", date(2022, 4, 20)),
        (5, "Echo Ltd", "West", date(2022, 5, 25)),
        (6, "Foxtrot GmbH", "West", date(2022, 6, 30)),
        (7, "Gamma SA", "East", date(2023, 1, 10)),
        (8, "Hotel BV", "East", date(2023, 2, 15)),
    ],
)

create_view(
    "orders",
    "order_id INT, customer_id INT, status STRING, amount INT, order_date DATE",
    [
        (1, 1, "completed", 100, date(2023, 1, 5)),
        (2, 1, "completed", 200, date(2023, 2, 10)),
        (3, 1, "completed", 150, date(2023, 3, 15)),
        (4, 2, "completed", 300, date(2023, 1, 20)),
        (5, 2, "cancelled", 250, date(2023, 2, 5)),
        (6, 2, "completed", 400, date(2023, 3, 20)),
        (7, 3, "completed", 120, date(2023, 1, 25)),
        (8, 3, "completed", 180, date(2023, 2, 28)),
        (9, 4, "pending", 220, date(2023, 2, 14)),
        (10, 4, "completed", 320, date(2023, 3, 30)),
        (11, 5, "completed", 500, date(2023, 1, 2)),
        (12, 5, "completed", 450, date(2023, 3, 5)),
        (13, 6, "completed", 275, date(2023, 2, 18)),
        (14, 6, "cancelled", 325, date(2023, 3, 25)),
    ],
)

### Checking the dimension is unique

Joining a fact table to a dimension must not change its row count. That only holds when the dimension key is unique.

In [ ]:
check(
    "SELECT COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_keys FROM customers",
    [(0,)],
    "dimension key is unique",
)
check(
    """
    SELECT COUNT(*) AS fact_rows
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    """,
    [(14,)],
    "a many-to-one join keeps the fact row count",
)

## Slowly changing dimensions (Type 2)

Customer attributes change over time, and history must be preserved. A Type 2 dimension stores one row per
version, with `effective_from` and `effective_to`. The interval is **half-open**: `[effective_from, effective_to)`.
A `NULL` end means the version is current.

To find the version that was valid at each transaction, use a range join. Half-open intervals make the boundary
unambiguous: a transaction at exactly `effective_to` belongs to the next version.

In [ ]:
create_view(
    "segment_history",
    "customer_id INT, segment STRING, effective_from TIMESTAMP, effective_to TIMESTAMP",
    [
        (1, "standard", datetime(2026, 1, 1, 0, 0), datetime(2026, 6, 1, 0, 0)),
        (1, "premium", datetime(2026, 6, 1, 0, 0), None),
        (2, "standard", datetime(2026, 1, 1, 0, 0), None),
    ],
)
create_view(
    "txns",
    "txn_id INT, customer_id INT, event_ts TIMESTAMP, amount DOUBLE",
    [
        (1, 1, datetime(2026, 1, 15, 12, 0), 20.0),
        (2, 1, datetime(2026, 7, 1, 12, 0), 30.0),
        (3, 2, datetime(2026, 3, 1, 12, 0), 40.0),
        (4, 99, datetime(2026, 3, 1, 12, 0), 50.0),
        (5, 1, datetime(2026, 6, 1, 0, 0), 10.0),
    ],
)

In [ ]:
check(
    """
    SELECT t.txn_id, s.segment
    FROM txns t
    LEFT JOIN segment_history s
      ON s.customer_id = t.customer_id
     AND t.event_ts >= s.effective_from
     AND (s.effective_to IS NULL OR t.event_ts < s.effective_to)
    ORDER BY t.txn_id
    """,
    [(1, "standard"), (2, "premium"), (3, "standard"), (4, None), (5, "premium")],
    "segment valid at each transaction",
)

The result has one row per transaction, including transaction 4, which has no matching customer. Transaction 5
falls exactly on the boundary and gets the new segment.

Before you trust a Type 2 table, check that no two versions for one key overlap. Any overlap would make the join
above return two rows for one transaction.

In [ ]:
check(
    """
    SELECT COUNT(*) AS overlapping_pairs
    FROM segment_history a
    JOIN segment_history b
      ON a.customer_id = b.customer_id
     AND a.effective_from < b.effective_from
     AND b.effective_from < COALESCE(a.effective_to, TIMESTAMP '9999-12-31 00:00:00')
    """,
    [(0,)],
    "no overlapping versions",
)

## Reading a plan

`EXPLAIN FORMATTED` shows how Spark will run a query. Look for the join strategy and for `Exchange` nodes, which
are shuffles across the network. A `BroadcastHashJoin` sends the small side to every executor and avoids a
shuffle of the large side.

Here the `BROADCAST` hint asks Spark to broadcast the small `customers` table. The result is the same either way;
the plan is what changes.

In [ ]:
BROADCAST_QUERY = """
    SELECT /*+ BROADCAST(c) */ c.region, SUM(o.amount) AS completed_total
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.region
    ORDER BY c.region
"""
check(
    BROADCAST_QUERY,
    [("North", 1150), ("South", 620), ("West", 1225)],
    "regional totals with a broadcast hint",
)
if SPARK_AVAILABLE:
    plan = spark.sql("EXPLAIN FORMATTED " + BROADCAST_QUERY).first()[0]
    assert "BroadcastHashJoin" in plan, plan
    print("PASS plan uses BroadcastHashJoin")
else:
    print("SKIPPED plan check: needs a Spark session.")

## Keep filters sargable

A predicate is *sargable* when the engine can use it to skip data, for example a comparison of a bare column
against a constant. Wrapping the column in a function forces a scan of every row. Compare a range on the date
with a function on the same date: both return February's five orders, but on a large partitioned table only the
range form can skip files.

In [ ]:
check(
    """
    SELECT COUNT(*) AS february_orders
    FROM orders
    WHERE order_date >= DATE '2023-02-01' AND order_date < DATE '2023-03-01'
    """,
    [(5,)],
    "range predicate on the bare column",
)
check(
    """
    SELECT COUNT(*) AS february_orders
    FROM orders
    WHERE date_format(order_date, 'yyyy-MM') = '2023-02'
    """,
    [(5,)],
    "function on the column gives the same answer",
)

## Nested values

Spark can keep collections and structs in one column. `collect_set` gathers distinct values into an array, and
`named_struct` builds a struct you can read a field from. They are useful for denormalised reporting tables.

In [ ]:
check(
    """
    SELECT customer_id, sort_array(collect_set(status)) AS statuses
    FROM orders
    WHERE customer_id = 2
    GROUP BY customer_id
    """,
    [(2, ["cancelled", "completed"])],
    "array of distinct statuses",
)
check(
    "SELECT named_struct('id', order_id, 'amount', amount).amount AS amount FROM orders WHERE order_id = 1",
    [(100,)],
    "field of a struct",
)

## Optimization habits

- Select only the columns you need. Columnar engines skip the rest.
- Filter early, with sargable predicates on bare columns.
- Broadcast the small side of a join, and check the plan with `EXPLAIN FORMATTED`.
- Watch the number of `Exchange` nodes. Each one is a shuffle, and most are caused by a join or an aggregate
  on a key that is not already partitioned that way.
- Measure a change on representative data. A plan that is faster on ten rows can be slower on ten billion.

## Takeaways

- Write the grain first, then make every key and join agree with it.
- Dimension keys must be unique. Check it.
- Model changing attributes as half-open Type 2 intervals, and test that they do not overlap.
- Read the plan before you tune. Spark will tell you which join it chose and where the shuffles are.